# Pricer de produits structurés électricité France

1. Lancer la cellule **Données** : synchronise power-hpfc avec GitHub, charge la dernière courbe, vérifie qu'elle
   reproduit tous les settlements EEX de la séance, charge l'historique spot et les paramètres.
2. Lancer la cellule **Formulaire** : décrire le deal (side, settlement, product, période, MW, grille semaine à cocher,
   hedging, margin), puis **Price**. **Save pricing** l'enregistre dans `pricings/`.

Les paramètres (frais, spread de financement, politique de marge) sont dans `config/pricer.yaml` et `config/marge.yaml`.

In [ ]:
from pathlib import Path

from IPython.display import display

from pricer.market import Market
from pricer.risk import history_bounds
from pricer.settings import Settings

HPFC_REPO = Path("~/Documents/2-Coding/python/GitHub/power-hpfc").expanduser()
MARKET = "FR"
PRICING_DATE = "latest"          # ou une date de séance archivée, ex. "2026-10-13"

mkt = Market.load(HPFC_REPO, MARKET, PRICING_DATE, sync=True)
settings = Settings.load()
spot = mkt.spot(*history_bounds(mkt, settings.history_start))
print(mkt.summary())
print(f"Spot day-ahead historique : {len(spot)} heures, du {spot.index[0].tz_convert(mkt.tz):%d/%m/%Y} "
      f"au {spot.index[-1].tz_convert(mkt.tz):%d/%m/%Y}")
todo = settings.missing(need_financing=True, manual_margin=False)
print("Paramètres à renseigner :", ", ".join(todo) if todo else "aucun")

In [ ]:
from pricer.ui import PricerForm

form = PricerForm(mkt, settings, spot)
display(form.widget)

## Pricing par le code (facultatif)

Les mêmes calculs sont accessibles directement, par exemple pour pricer une série de deals.

In [ ]:
from datetime import date

from pricer.products import grid_from_hours, weekly
from pricer.quote import quote
from pricer.ui import show

deal = weekly(mkt, date(2027, 1, 1), date(2027, 12, 31), 5, grid_from_hours(range(6, 22), range(5)), "sell")
show(quote(mkt, deal, settings, spot=spot))

## Diagnostic : la forme de la HPFC face au spot réalisé (facultatif)

Pour chaque courbe archivée du jour J, TBx de la HPFC sur J+1 contre TBx du spot réalisé de J+1.
La HPFC doit être un plancher du spot en moyenne (convexité du TBx).

In [ ]:
import pandas as pd
from pricer.tbx import TB_HOURS, backtest

bt = backtest(mkt)
print(f"{len(bt)} jours livrés ; base moyenne HPFC {bt['HPFC base'].mean():.2f}, spot {bt['spot base'].mean():.2f}")
display(pd.DataFrame({"HPFC": [bt[f"HPFC TB{x}"].mean() for x in TB_HOURS],
                      "spot": [bt[f"spot TB{x}"].mean() for x in TB_HOURS]},
                     index=[f"TB{x}" for x in TB_HOURS]).round(2))